# 02 — Anomalies and EOF Decomposition

Compute weekly anomalies, run EOF on SST and SSH, compute NAO and ENSO indices.


In [ ]:
%load_ext autoreload
%autoreload 2
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt

from src.preprocessing.anomalies import process_and_save
from src.eof.compute_eofs import compute_eofs, load_eofs
from src.indices.nao  import compute_nao, load_nao
from src.indices.enso import compute_enso, load_enso

In [ ]:
# Run the pipeline (or load cached output if already written).
# process_and_save('sst'); process_and_save('ssh')
# compute_eofs('sst'); compute_eofs('ssh')
sst_eof = load_eofs('sst')
ssh_eof = load_eofs('ssh')
print('SST modes retained:', sst_eof['n_modes'])
print('SSH modes retained:', ssh_eof['n_modes'])

In [ ]:
# Plot cumulative explained variance
for eof_data, label in [(sst_eof, 'SST'), (ssh_eof, 'SSH')]:
    cum = np.cumsum(eof_data['variance_fraction'])
    plt.plot(range(1, len(cum)+1), cum, marker='o', label=label)
plt.axhline(0.90, color='k', linestyle='--')
plt.xlabel('Mode'); plt.ylabel('Cumulative variance explained')
plt.legend(); plt.show()

In [ ]:
# Plot first 4 SSH EOF modes
eof_data = ssh_eof
lats, lons = eof_data['lat'], eof_data['lon']
mask = eof_data['ocean_mask']
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for k, ax in enumerate(axes.flat):
    pattern = np.full(mask.shape, np.nan)
    pattern[mask] = eof_data['eofs'][k]
    im = ax.pcolormesh(lons, lats, pattern, cmap='RdBu_r', shading='auto')
    ax.set_title(f'SSH EOF mode {k+1} ({100*eof_data["variance_fraction"][k]:.1f}%)')
    plt.colorbar(im, ax=ax, shrink=0.7)

In [ ]:
# Climate indices
nao  = load_nao()
enso = load_enso()
fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
ax[0].plot(nao.index, nao.values); ax[0].set_title('NAO (weekly)')
ax[1].plot(enso.index, enso.values); ax[1].set_title('Niño 3.4 (weekly)')